# XGBoost 재튜닝 (Optuna 30trial + 3-Fold 탐색 → 5-Fold 최종 학습)

torch를 전혀 사용하지 않는 독립 노트북입니다. CatBoost 재튜닝 노트북과 완전히
같은 구조라서, 따로 스레드 충돌 걱정 없이 정상 속도로 동작합니다.

**사전 준비**: `uv add xgboost`(이미 설치하셨다면 생략)

**예상 시간**: 탐색 단계(30trial × 3-fold) + 최종 5-fold 학습 합쳐서 대략 20~40분

## 1. 라이브러리 및 설정

In [1]:
import os
import json
import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = "../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_TRIALS = 30
SEARCH_SPLITS = 3
FINAL_SPLITS = 5
RANDOM_STATE = 1004
CACHE_DIR = "blend_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

## 2. 데이터 전처리 (XGBoost 네이티브 범주형 사용)

In [2]:
train_raw = pd.read_csv(TRAIN_PATH).drop(columns=["ID"])


def base_features(df):
    df = df.copy()
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)
    return df


def fill_na(df):
    cat_cols = df.select_dtypes(include=["object"]).columns
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns
    na_cols = df.isna().sum().loc[lambda x: x > 0].index
    for col in na_cols:
        if col in cat_cols:
            df[col] = df[col].fillna("해당없음")
        elif col in num_cols:
            df[col] = df[col].fillna(-1)
    return df


df = fill_na(base_features(train_raw.copy()).drop(columns=DEAD_COLS))

# XGBoost는 enable_categorical=True + pandas category dtype을 쓰면
# 라벨인코딩 없이 범주형을 네이티브로 처리할 수 있습니다 (보통 성능이 더 좋음).
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    df[col] = df[col].astype("category")

y = df[TARGET_COL].values.astype(np.float32)
X = df.drop(columns=[TARGET_COL])

print(f"전처리 완료: {X.shape}, 범주형 피처 {len(cat_cols)}개")

전처리 완료: (256351, 67), 범주형 피처 20개


## 3. Optuna 탐색 (3-Fold, 30 trial)

In [3]:
def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    }

    skf = StratifiedKFold(n_splits=SEARCH_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = []
    for tr_idx, val_idx in skf.split(X, y):
        model = xgb.XGBClassifier(
            n_estimators=1500,
            **params,
            tree_method="hist",
            enable_categorical=True,
            eval_metric="auc",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            early_stopping_rounds=50,
        )
        model.fit(
            X.iloc[tr_idx], y[tr_idx],
            eval_set=[(X.iloc[val_idx], y[val_idx])],
            verbose=False,
        )
        preds = model.predict_proba(X.iloc[val_idx])[:, 1]
        fold_scores.append(roc_auc_score(y[val_idx], preds))

    return float(np.mean(fold_scores))


def print_progress(study, trial):
    print(f"Trial {trial.number + 1}/{N_TRIALS}  AUC={trial.value:.5f}  (현재 최고: {study.best_value:.5f})")


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=N_TRIALS, callbacks=[print_progress])

print("\n최적 파라미터:")
print(study.best_params)
print(f"탐색 단계 최고 AUC (3-fold): {study.best_value:.5f}")

Trial 1/30  AUC=0.73865  (현재 최고: 0.73865)
Trial 2/30  AUC=0.73760  (현재 최고: 0.73865)
Trial 3/30  AUC=0.73863  (현재 최고: 0.73865)
Trial 4/30  AUC=0.73723  (현재 최고: 0.73865)
Trial 5/30  AUC=0.73906  (현재 최고: 0.73906)
Trial 6/30  AUC=0.73954  (현재 최고: 0.73954)
Trial 7/30  AUC=0.73812  (현재 최고: 0.73954)
Trial 8/30  AUC=0.73808  (현재 최고: 0.73954)
Trial 9/30  AUC=0.73928  (현재 최고: 0.73954)
Trial 10/30  AUC=0.73620  (현재 최고: 0.73954)
Trial 11/30  AUC=0.73835  (현재 최고: 0.73954)
Trial 12/30  AUC=0.73926  (현재 최고: 0.73954)
Trial 13/30  AUC=0.73951  (현재 최고: 0.73954)
Trial 14/30  AUC=0.73943  (현재 최고: 0.73954)
Trial 15/30  AUC=0.73963  (현재 최고: 0.73963)
Trial 16/30  AUC=0.73959  (현재 최고: 0.73963)
Trial 17/30  AUC=0.73920  (현재 최고: 0.73963)
Trial 18/30  AUC=0.73892  (현재 최고: 0.73963)
Trial 19/30  AUC=0.73911  (현재 최고: 0.73963)
Trial 20/30  AUC=0.73937  (현재 최고: 0.73963)
Trial 21/30  AUC=0.73962  (현재 최고: 0.73963)
Trial 22/30  AUC=0.73939  (현재 최고: 0.73963)
Trial 23/30  AUC=0.73909  (현재 최고: 0.73963)
Trial 24/30  AUC=0.7

## 4. 최적 파라미터로 5-Fold 최종 학습

In [4]:
best_params = study.best_params

skf = StratifiedKFold(n_splits=FINAL_SPLITS, shuffle=True, random_state=RANDOM_STATE)
oof_xgboost = np.zeros(len(y))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y), start=1):
    model = xgb.XGBClassifier(
        n_estimators=3000,
        **best_params,
        tree_method="hist",
        enable_categorical=True,
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=100,
    )
    model.fit(
        X.iloc[tr_idx], y[tr_idx],
        eval_set=[(X.iloc[val_idx], y[val_idx])],
        verbose=False,
    )
    oof_xgboost[val_idx] = model.predict_proba(X.iloc[val_idx])[:, 1]
    fold_auc = roc_auc_score(y[val_idx], oof_xgboost[val_idx])
    print(f"Fold {fold}/{FINAL_SPLITS}  XGBoost AUC: {fold_auc:.5f}  (best_iteration: {model.best_iteration})")

score_xgboost = roc_auc_score(y, oof_xgboost)
print(f"\nXGBoost 5-Fold 전체 OOF AUC: {score_xgboost:.5f}")

Fold 1/5  XGBoost AUC: 0.74364  (best_iteration: 1123)
Fold 2/5  XGBoost AUC: 0.74052  (best_iteration: 1326)
Fold 3/5  XGBoost AUC: 0.73765  (best_iteration: 906)
Fold 4/5  XGBoost AUC: 0.73825  (best_iteration: 812)
Fold 5/5  XGBoost AUC: 0.73892  (best_iteration: 924)

XGBoost 5-Fold 전체 OOF AUC: 0.73972


## 5. 결과 저장

In [5]:
with open("xgboost_best_params.json", "w", encoding="utf-8") as f:
    json.dump(best_params, f, ensure_ascii=False, indent=2)

np.save(f"{CACHE_DIR}/oof_xgboost.npy", oof_xgboost)
np.save(f"{CACHE_DIR}/oof_y.npy", y.astype(np.float32))

print("저장 완료: xgboost_best_params.json, blend_cache/oof_xgboost.npy")
print(f"\nXGBoost 재튜닝 점수 = {score_xgboost:.5f}")

저장 완료: xgboost_best_params.json, blend_cache/oof_xgboost.npy

XGBoost 재튜닝 점수 = 0.73972


## 6. (보너스) 우리 3종(LightGBM튜닝+CatBoost+XGBoost) 앙상블 빠르게 확인

In [6]:
try:
    oof_lgbm_tuned = np.load(f"{CACHE_DIR}/oof_lgbm_tuned.npy")
    oof_catboost = np.load(f"{CACHE_DIR}/oof_catboost.npy")

    score_lgbm = roc_auc_score(y, oof_lgbm_tuned)
    score_cat = roc_auc_score(y, oof_catboost)

    print(f"LightGBM(튜닝): {score_lgbm:.5f}")
    print(f"CatBoost(재튜닝): {score_cat:.5f}")
    print(f"XGBoost(재튜닝): {score_xgboost:.5f}")
    print()

    # 단순 가중평균 그리드서치 (3종, 0.1 단위)
    best_combo, best_score = None, max(score_lgbm, score_cat, score_xgboost)
    step = 0.1
    weights = [round(i * step, 1) for i in range(int(1 / step) + 1)]
    for w1 in weights:
        for w2 in weights:
            w3 = round(1 - w1 - w2, 2)
            if w3 < 0 or w3 > 1:
                continue
            blend = w1 * oof_lgbm_tuned + w2 * oof_catboost + w3 * oof_xgboost
            blend_score = roc_auc_score(y, blend)
            if blend_score > best_score:
                best_score = blend_score
                best_combo = (w1, w2, w3)

    print(f"3종 가중평균 최적 조합: LightGBM {best_combo[0]} + CatBoost {best_combo[1]} + XGBoost {best_combo[2]}")
    print(f"점수: {best_score:.5f}  (단독 최고 대비 개선: {best_score - max(score_lgbm, score_cat, score_xgboost):+.5f})")
    print()
    print(f"참고: 김영혜님 v5 3종 앙상블(0.4/0.3/0.3) OOF는 0.74043이었습니다.")
except FileNotFoundError as e:
    print(f"필요한 파일이 없습니다: {e}")
    print("lgbm_tuned_oof_only.ipynb를 먼저 실행해서 oof_lgbm_tuned.npy를 만들어주세요.")

LightGBM(튜닝): 0.73998
CatBoost(재튜닝): 0.73971
XGBoost(재튜닝): 0.73972

3종 가중평균 최적 조합: LightGBM 0.4 + CatBoost 0.3 + XGBoost 0.3
점수: 0.74027  (단독 최고 대비 개선: +0.00029)

참고: 김영혜님 v5 3종 앙상블(0.4/0.3/0.3) OOF는 0.74043이었습니다.
